In [1]:
import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city:str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return(
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}degree celsius\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [7]:
result = search_tool.invoke("Give me the latest news on CJP?")
result

[{'url': 'https://www.thehindu.com/videos/cjp-leaders-bail-hearing-delayed-us-lawmakers-raise-concerns-over-protest-crackdown/article71602319.ece',
  'content': 'The bail hearing of the Cockroach Janta Party (CJP) co-convenors Saurav Das and Ashutosh Ranka was delayed on Sunday, October 18, according to the party’s legal head, Ratna Singh. The two leaders were sent to five days of judicial custody after entering Jantar Mantar by scaling its locked gates, following the denotification of the site as a designated protest venue.\n\nUpdated  - October 19, 2026 01:09 pm IST\n\nRemove  SEE ALL\n\n Copy link\n Email\n Facebook\n Twitter\n Telegram\n LinkedIn\n WhatsApp\n Reddit\n\nPublished - October 19, 2026 01:01 pm IST\n\n## Latest\n\n[](\n\nVideos \n\n### Beyond religiosity: on the Kolkata pandal tragedy| The Hindu Editorial\n\n[](\n\nVideos \n\n### CJP leaders’ bail hearing “delayed”; U.S. lawmakers raise concerns over protest crackdown\n\n[](\n\nVideos'},
 {'url': 'https://www.bbc.com/ne

In [8]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [10]:
response = llm.invoke("Tell me a joke about tomatoes?")
response.content

'Why did the tomato turn red?\n\nBecause it saw the salad dressing! 🍅😄'

In [11]:
tools = [search_tool,get_weather_data]

In [12]:
tool_calling_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant. "
        "Use the available tools whenever they are needed. "
        "Provide accurate, concise answers based on the tool results."
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [13]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=tool_calling_prompt
)

In [14]:
# EXECUTOR

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [16]:
response = agent_executor.invoke({
    "input":(
        "Find the capital of Russia"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...

Invoking: `get_weather_data` with `{'city': 'Moscow'}`


City: Moscow
Temperature: 11degree celsius
Weather: Overcast 
Humidity: 25%The capital of Russia is **Moscow**.  

**Current weather in Moscow (as of the latest report):**  
- **Temperature:** 11 °C  
- **Condition:** Overcast  
- **Humidity:** 25%

> Finished chain.


In [17]:
print(response["output"])

The capital of Russia is **Moscow**.  

**Current weather in Moscow (as of the latest report):**  
- **Temperature:** 11 °C  
- **Condition:** Overcast  
- **Humidity:** 25%
